In [ ]:
import numpy as np
import adaptive_latents
from adaptive_latents import datasets
from sim_stim import make_srs, make_slices_tensor
import pathlib
from learn_s_hat_plots import plot_onestep_pred_error_decreasing, make_table
import matplotlib.pyplot as plt

In [ ]:
rng = np.random.default_rng(0)
d = datasets.Zong22Dataset()
data = d.neural_data

srs = make_srs(data, rng, comparison_preset='visualization', n_runs=1, show_tqdm=True, overrides=dict(stim_magnitude = 20_000, decay_rate=.8, smoothing_tau=.7))


In [ ]:
%matplotlib inline
i= 23
sr = srs['learning from stim'][0]

fig2, axs2 = plt.subplots(ncols=4, figsize=(16,4), sharex=False, sharey=False, layout='constrained')
latents = sr.log['latents'].slice_by_time(slice(30,None))
axs2[0].plot(latents[:, 0], latents[:, 1])
stim_s = sr.log['stim_intended_samples'].t - latents.dt
axs2[0].plot(latents.slice_by_time(stim_s)[:, 0], latents.slice_by_time(stim_s)[:, 1], '.', color='r')

l = .5
r = 2
center_t = sr.log['stim_intended_samples'].t[i]
latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
axs2[1].plot(latents[:, 0], latents[:, 1])
stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
axs2[1].plot(latents_s[:, 0], latents_s[:, 1], '.', color='r')

axs2[2].plot(latents.t, latents);
for stim_t in stim_s:
    axs2[2].axvline(stim_t, color='r')



# u = sr.stim_designer.log[i]['pro'].Q[:,0]
u = sr.stim_designer.log[i]['u']
idx = np.argsort(np.abs(u))[::-1]
print()

high_d = sr.log['high_d_with_stim'].slice_by_time(slice(center_t-l,center_t+r))
axs2[3].plot(high_d.t, high_d[:,idx[:int(np.linalg.norm(u,ord=0))]]);
for stim_t in stim_s:
    axs2[3].axvline(stim_t, color='r')



In [ ]:
fig2.savefig(pathlib.Path('/home/jgould/Documents/neurips_2025/generated/zong_stim.svg'))


In [ ]:
# %matplotlib qt
# i = 25
# r = 10
# idx = np.argsort(u)
#
#
# center_t = srs['learning from stim'][0].log['stim_intended_samples'].t[i]
# high_d_data = srs['learning from stim'][0].log['high_d_with_stim'].slice_by_time(slice(center_t-r,center_t+r))
# high_d_data_nostim = srs['learning from stim'][0].log['high_d_without_stim'].slice_by_time(slice(center_t-r,center_t+r))
#
#
# fig, axs = plt.subplots(nrows=2)
#
# ax = axs[0]
# ax.plot(high_d_data.t, high_d_data[:, idx[-10:]], color='gray')
#
# ax = axs[1]
# ax.plot(high_d_data_nostim.t, high_d_data_nostim[:, idx[-10:]], color='gray')
#
# for ax in axs:
#     for center_t in srs['learning from stim'][0].log['stim_intended_samples'].slice_by_time(slice(center_t-r,center_t+r)).t:
#         ax.axvline(center_t - high_d_data.dt, color='r')
# # ax.set_xlim()

In [ ]:
high_d_data.shape